In [154]:
import pandas as pd
df = pd.read_csv('100_Unique_QA_Dataset.csv')

In [155]:
df.head()

,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100


In [156]:
# tokenize
def tokenize(text):
  text = text.lower()
  text = text.replace("?", "")
  text = text.replace("'", "")
  return text.split()


In [157]:
# Vocabulary
vocab = {'<Unknown>': 0}

In [158]:
def build_vocab(row):
  tokenized_question = tokenize(row['question'])
  tokenized_answer = tokenize(row['answer'])
  merged_tokens = tokenized_question + tokenized_answer
  print(merged_tokens)

  for token in merged_tokens:
    if token not in vocab:
      vocab[token] = len(vocab)

In [159]:
df.apply(build_vocab, axis=1)

['what', 'is', 'the', 'capital', 'of', 'france', 'paris']
['what', 'is', 'the', 'capital', 'of', 'germany', 'berlin']
['who', 'wrote', 'to', 'kill', 'a', 'mockingbird', 'harper-lee']
['what', 'is', 'the', 'largest', 'planet', 'in', 'our', 'solar', 'system', 'jupiter']
['what', 'is', 'the', 'boiling', 'point', 'of', 'water', 'in', 'celsius', '100']
['who', 'painted', 'the', 'mona', 'lisa', 'leonardo-da-vinci']
['what', 'is', 'the', 'square', 'root', 'of', '64', '8']
['what', 'is', 'the', 'chemical', 'symbol', 'for', 'gold', 'au']
['which', 'year', 'did', 'world', 'war', 'ii', 'end', '1945']
['what', 'is', 'the', 'longest', 'river', 'in', 'the', 'world', 'nile']
['what', 'is', 'the', 'capital', 'of', 'japan', 'tokyo']
['who', 'developed', 'the', 'theory', 'of', 'relativity', 'albert-einstein']
['what', 'is', 'the', 'freezing', 'point', 'of', 'water', 'in', 'fahrenheit', '32']
['which', 'planet', 'is', 'known', 'as', 'the', 'red', 'planet', 'mars']
['who', 'is', 'the', 'author', 'of', '19

,0
0,None
1,None
2,None
3,None
4,None
...,...
85,None
86,None
87,None
88,None


In [160]:
# length to unique words in dataset
len(vocab)

324

In [161]:
# Converts words to numerical indices
def text_to_indices(text, vocab):
  indexed_text = []
  for token in tokenize(text):
    if token in vocab:
      indexed_text.append(vocab[token])
    else:
      indexed_text.append(vocab['<Unknown>'])
  return indexed_text

In [162]:
import torch
from torch.utils.data import Dataset, DataLoader

In [163]:
class QnA(Dataset):
  def __init__(self, df, vocab):
    self.df = df
    self.vocab = vocab

  def __len__(self):
    return self.df.shape[0]

  def __getitem__(self, index):
    numerical_question = text_to_indices(self.df.iloc[index]['question'], self.vocab)
    numerical_answer = text_to_indices(self.df.iloc[index]['answer'], self.vocab)

    return torch.tensor(numerical_question), torch.tensor(numerical_answer)

In [164]:
dataset = QnA(df, vocab)

In [165]:
dataset[1]

(tensor([1, 2, 3, 4, 5, 8]), tensor([9]))

In [166]:
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

In [167]:
for question, answer in dataloader:
  print(question, answer)

tensor([[ 10, 140,   3, 141, 270,  93, 271,   5,   3, 272]]) tensor([[273]])
tensor([[  1,   2,   3, 141, 117,  83,   3, 277, 278]]) tensor([[121]])
tensor([[  1,   2,   3,  17, 115,  83,  84]]) tensor([[116]])
tensor([[ 42,   2,   3, 274, 211, 275]]) tensor([[276]])
tensor([[ 42, 107,   2, 108,  19, 109]]) tensor([[110]])
tensor([[ 78,  79, 129,  81,  19,   3,  21,  22]]) tensor([[36]])
tensor([[  1,   2,   3,  69,   5, 155]]) tensor([[156]])
tensor([[  1,   2,   3, 180, 181, 182, 183]]) tensor([[184]])
tensor([[  1,   2,   3,   4,   5, 279]]) tensor([[280]])
tensor([[  1,   2,   3,  33,  34,   5, 245]]) tensor([[246]])
tensor([[10, 55,  3, 56,  5, 57]]) tensor([[58]])
tensor([[ 1,  2,  3, 92, 93, 94]]) tensor([[95]])
tensor([[ 1,  2,  3, 50, 51, 19,  3, 45]]) tensor([[52]])
tensor([[ 42, 137,   2,  62,  39,   3, 322, 323]]) tensor([[6]])
tensor([[ 10,  75,   3, 296,  19, 297]]) tensor([[298]])
tensor([[  1,   2,   3,   4,   5, 135]]) tensor([[136]])
tensor([[10, 96,  3, 97]]) tensor(

In [168]:
import torch.nn as nn

In [169]:
class SimpleRNN(nn.Module):
  def __init__(self, vocab_size):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, embedding_dim=50)
    self.rnn = nn.RNN(50, 64, batch_first=True)
    self.fc = nn.Linear(64, vocab_size)

  def forward(self, question):
    embedded_question = self.embedding(question)
    hidden, final = self.rnn(embedded_question)
    output = self.fc(final.squeeze(0))
    return output


In [170]:
x = nn.Embedding(324, embedding_dim=50)
y = nn.RNN(50, 64, batch_first=True)
z = nn.Linear(64, 324)

a = dataset[0][0].reshape(1, 6)
print(a.shape)
b = x(a)
print(b.shape)
c, d = y(b)
print(c.shape)
print(d.shape)
e = z(d.squeeze(0))
print(e.shape)

torch.Size([1, 6])
torch.Size([1, 6, 50])
torch.Size([1, 6, 64])
torch.Size([1, 1, 64])
torch.Size([1, 324])


In [171]:
learning_rate = 0.001
epochs = 20

In [172]:
model = SimpleRNN(len(vocab))

In [173]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [174]:
# training loop
for epoch in range(epochs):
  total_loss = 0

  for question, answer in dataloader:
    optimizer.zero_grad()

    # forward pass
    prediction = model(question)

    # loss
    loss = criterion(prediction, answer[0])

    # calculate gradients
    loss.backward()

    # update
    optimizer.step()

    total_loss += loss.item()

  print(f'Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(dataloader)}')

Epoch 1/20, Loss: 5.847534269756741
Epoch 2/20, Loss: 5.07504629029168
Epoch 3/20, Loss: 4.182002499368456
Epoch 4/20, Loss: 3.5009381824069554
Epoch 5/20, Loss: 2.9352005865838793
Epoch 6/20, Loss: 2.4133021765285068
Epoch 7/20, Loss: 1.9318919393751357
Epoch 8/20, Loss: 1.5231712758541107
Epoch 9/20, Loss: 1.1832094003756841
Epoch 10/20, Loss: 0.9117288639148077
Epoch 11/20, Loss: 0.7137459890709983
Epoch 12/20, Loss: 0.5524749689631991
Epoch 13/20, Loss: 0.44437930120362173
Epoch 14/20, Loss: 0.36212852978044086
Epoch 15/20, Loss: 0.298188216984272
Epoch 16/20, Loss: 0.25041297723849615
Epoch 17/20, Loss: 0.2098772753443983
Epoch 18/20, Loss: 0.18224740400910378
Epoch 19/20, Loss: 0.1569875122772323
Epoch 20/20, Loss: 0.13754121545288298


In [177]:
def predict(model, question, threshold=0.5):
  numerical_question = text_to_indices(question, vocab)
  question_tensor = torch.tensor(numerical_question).unsqueeze(0)

  output = model(question_tensor)
  probabilities = torch.softmax(output, dim=1)
  value, index = torch.max(probabilities, dim=1)
  print(value, index)

  if value < threshold:
    return "I don't know"
  else:
    return list(vocab.keys())[list(vocab.values()).index(index.item())]

In [179]:
predict(model, "What is the capital of france?")

tensor([0.8893], grad_fn=<MaxBackward0>) tensor([7])


'paris'